# CI/CD Build Data - Exploratory Analysis

This notebook explores build data collected from CircleCI to understand patterns and prepare for ML model training.

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import DataLoader
from src.feature_engineering import FeatureEngineer

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries imported successfully!')

## 1. Load Data

In [ ]:
# Load data from MongoDB
loader = DataLoader()
loader.connect()

df = loader.load_training_data(days=30)
print(f"Loaded {len(df)} builds")
df.head()

## 2. Data Summary

In [ ]:
# Basic statistics
print("Dataset Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nMissing Values:")
print(df.isnull().sum())
print("\nData Types:")
print(df.dtypes)

## 3. Build Status Distribution

In [ ]:
# Status distribution
if 'status' in df.columns:
    status_counts = df['status'].value_counts()
    
    plt.figure(figsize=(10, 6))
    status_counts.plot(kind='bar', color=['green', 'red', 'yellow'])
    plt.title('Build Status Distribution')
    plt.xlabel('Status')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.show()
    
    print("\nStatus Distribution:")
    print(status_counts)
    print(f"\nSuccess Rate: {(status_counts.get('success', 0) / len(df)) * 100:.2f}%")

## 4. Build Duration Analysis

In [ ]:
# Duration analysis
if 'duration' in df.columns:
    df_with_duration = df[df['duration'].notna()].copy()
    df_with_duration['duration_minutes'] = df_with_duration['duration'] / 60
    
    plt.figure(figsize=(14, 6))
    
    plt.subplot(1, 2, 1)
    df_with_duration['duration_minutes'].hist(bins=30, edgecolor='black')
    plt.title('Build Duration Distribution')
    plt.xlabel('Duration (minutes)')
    plt.ylabel('Frequency')
    
    plt.subplot(1, 2, 2)
    df_with_duration.boxplot(column='duration_minutes', by='status')
    plt.title('Duration by Status')
    plt.suptitle('')
    plt.xlabel('Status')
    plt.ylabel('Duration (minutes)')
    
    plt.tight_layout()
    plt.show()
    
    print("\nDuration Statistics:")
    print(df_with_duration['duration_minutes'].describe())

## 5. Temporal Patterns

In [ ]:
# Time-based analysis
if 'created_at' in df.columns:
    df['created_at'] = pd.to_datetime(df['created_at'])
    df['hour'] = df['created_at'].dt.hour
    df['day_of_week'] = df['created_at'].dt.day_name()
    
    plt.figure(figsize=(14, 6))
    
    plt.subplot(1, 2, 1)
    df['hour'].value_counts().sort_index().plot(kind='bar')
    plt.title('Builds by Hour of Day')
    plt.xlabel('Hour')
    plt.ylabel('Number of Builds')
    
    plt.subplot(1, 2, 2)
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    df['day_of_week'].value_counts()[day_order].plot(kind='bar')
    plt.title('Builds by Day of Week')
    plt.xlabel('Day')
    plt.ylabel('Number of Builds')
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()

## 6. Feature Engineering

In [ ]:
# Apply feature engineering
engineer = FeatureEngineer()
df_features = engineer.create_training_features(df)

print(f"\nFeatures extracted: {len(df_features.columns)} columns")
print("\nNew feature columns:")
print([col for col in df_features.columns if col not in df.columns])

## 7. Correlation Analysis

In [ ]:
# Feature correlations
numeric_cols = df_features.select_dtypes(include=[np.number]).columns
correlation = df_features[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation, cmap='coolwarm', center=0, annot=False)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# Show top correlations with failure
if 'is_failed' in correlation.columns:
    print("\nTop features correlated with failure:")
    print(correlation['is_failed'].sort_values(ascending=False).head(10))

## 8. Error Analysis

In [ ]:
# Error type distribution
error_cols = [col for col in df_features.columns if col.startswith('error_')]

if error_cols:
    error_data = df_features[error_cols].sum().sort_values(ascending=False)
    
    plt.figure(figsize=(10, 6))
    error_data.plot(kind='bar')
    plt.title('Error Type Distribution')
    plt.xlabel('Error Type')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 9. Prepare Training Data

In [ ]:
# Prepare features for model training
X, y = engineer.prepare_for_training(df_features, target_column='is_failed')

if X is not None and y is not None:
    print(f"Training data shape: {X.shape}")
    print(f"Target distribution:\n{y.value_counts()}")
    print(f"\nFeatures: {list(X.columns)}")
    
    # Save processed data
    df_features.to_csv('../data/processed/training_data.csv', index=False)
    print("\nProcessed data saved to data/processed/training_data.csv")

## 10. Insights Summary

In [ ]:
print("="*60)
print("KEY INSIGHTS")
print("="*60)

if 'status' in df.columns:
    success_rate = (df['status'] == 'success').mean() * 100
    print(f"1. Overall Success Rate: {success_rate:.2f}%")

if 'duration' in df.columns:
    avg_duration = df['duration'].mean() / 60
    print(f"2. Average Build Duration: {avg_duration:.1f} minutes")

if 'project_slug' in df.columns:
    print(f"3. Number of Projects: {df['project_slug'].nunique()}")

print(f"4. Total Builds Analyzed: {len(df)}")
print(f"5. Date Range: {df['created_at'].min()} to {df['created_at'].max()}")

print("\nReady for model training!")

In [ ]:
# Cleanup
loader.close()